# Part 2 — Fine-Tuning Diffusion Models for Domain-Specific Image Generation

In [ ]:
import subprocess, sys, random, numpy as np, torch
SEED = 266; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True,
    text=True).stdout.strip()
print({"commit": commit, "python": sys.version.split()[0], "torch": torch.__version__,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", "seed": SEED})


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))


## 2.1 Data Preparation

In [ ]:
from src.part2.data_prep import prepare_dataset
report = prepare_dataset("../data/processed/part2")
report


## 2.2 Fine-Tuning — two LoRA runs (rank 4 and rank 16), same data and step budget

In [ ]:
from src.part2.train_runs import run_training
run1_report = run_training("run1_lora_r4")
run1_report


In [ ]:
run2_report = run_training("run2_lora_r16")
run2_report


In [ ]:
import pandas as pd
pd.DataFrame([run1_report, run2_report])


## 2.3 Image Generation — base model + both runs, 20 fixed prompts x 2 seeds

In [ ]:
from src.part2.generate import generate_all
records = generate_all("../eval/part2_prompts.json", "../data/processed/part2/generated")
len(records)


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import json

with open("../data/processed/part2/generated/generation_metadata.json") as f:
    meta = json.load(f)

prompts = sorted(set(r["prompt"] for r in meta))[:8]
fig, axes = plt.subplots(len(prompts), 3, figsize=(9, 3 * len(prompts)))
models = ["base", "run1_lora_r4", "run2_lora_r16"]
for row, prompt in enumerate(prompts):
    for col, model in enumerate(models):
        record = next(r for r in meta if r["prompt"] == prompt and r["model"] == model and r["seed"] == 0)
        img = Image.open(f"../data/processed/part2/generated/{model}/{record['file_name']}")
        axes[row][col].imshow(img)
        axes[row][col].set_title(model, fontsize=8)
        axes[row][col].axis("off")
plt.tight_layout()
plt.show()


## 2.4 Evaluation, Safety & Documentation

In [ ]:
from src.part2.evaluate import evaluate_model
results = []
for model_name in models:
    results.append(evaluate_model(model_name, f"../data/processed/part2/generated/{model_name}", meta, "../data/processed/part2/test"))
pd.DataFrame(results)


In [ ]:
from src.part2.evaluate import memorization_check
in_domain_files = [r["file_name"] for r in meta if r["model"] == "run1_lora_r4" and r["category"] == "in_domain"]
pairs = memorization_check("../data/processed/part2/generated/run1_lora_r4", "../data/processed/part2/train", in_domain_files)
pairs


### Failure modes\nTODO after generation: show 2 example failure modes (e.g. ignoring the prompt on out-of-domain prompts, overfitting, repeated outputs) with images, and state which run is chosen as final and why.\n\n### Safety (C4)\nDataset license: lambdalabs/naruto-blip-captions, non-commercial research use (Naruto is a copyrighted character). NSFW filter: Stable Diffusion's built-in safety checker is enabled for any externally-facing generation. Memorization check: see closest-pair table above.

## Push to Hugging Face\nTODO: `huggingface-cli login`, then push both LoRA runs and the generated-image dataset, with a model card for the selected final run.